## Setup

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
# from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
# from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [3]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [4]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset(
    'wikiann',
    wikiann_label_map,
    lang='sk'
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [5]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'O', 'B-ORG', 'B-PER', 'I-ORG', 'I-LOC', 'I-PER', 'B-LOC'}


## conll2003-SK-NER
ju-bezdek/conll2003-SK-NER

In [6]:
conll2003_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-MISC": 7,
    "I-MISC": 8
}

conll2003 = ner.ReadNERData()
conll2003_words, conll2003_labels = wikiann.read_dataset(
    'ju-bezdek/conll2003-SK-NER',
    conll2003_label_map,
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/3453 [00:00<?, ?it/s]

In [7]:
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC
dataset_label_alignment = {
    'I-MISC': 'O',
    'B-MISC': 'O'
}

In [8]:
# Align the dataset labels to the standard labels
conll2003_labels = ner.align_dataset(conll2003_labels, dataset_label_alignment)

# Evaluate model

In [14]:
alignment = {
0: '0', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'
}

model_name = "crabz/slovakbert-ner"
model_name_output = 'slovakbert-ner'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

In [15]:
print(model_evaluation.model.config.id2label)

{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}


### wikiann

In [16]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` param

In [17]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.9370,0.9523,0.9446,4906
1,ORG,0.9010,0.8950,0.8980,3773
2,PER,0.9564,0.9715,0.9639,4601
3,_,0.0000,0.0000,0.0000,0
4,micro,0.5475,0.9427,0.6927,13280
5,macro,0.6986,0.7047,0.7016,13280
6,weighted,0.9335,0.9427,0.9380,13280


In [18]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,0,0.0000,0.0000,0.0000,0
1,B-LOC,0.9455,0.9588,0.9521,4906
2,B-ORG,0.9308,0.9194,0.9251,3773
3,B-PER,0.9653,0.9785,0.9718,4601
4,I-LOC,0.9488,0.9347,0.9417,4027
5,I-ORG,0.9503,0.9388,0.9445,6993
6,I-PER,0.9675,0.9752,0.9713,6039
7,O,0.0000,0.0000,0.0000,50267
8,accuracy,0.3584,80606,None,None
9,macro,0.7135,0.7132,0.7133,80606


## Conll2003

In [19]:
data_name = "conll2003"
conll_evaluation_output = model_evaluation.evaluate_model(conll2003_words, conll2003_labels)

  0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` param

In [20]:
conll_seqeval = conll_evaluation_output.get_classification('Seqeval')
conll_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.5186,0.6195,0.5646,1556
1,ORG,0.3309,0.4894,0.3948,1598
2,PER,0.7567,0.7155,0.7355,1543
3,_,0.0000,0.0000,0.0000,0
4,micro,0.3027,0.6068,0.4039,4697
5,macro,0.4015,0.4561,0.4237,4697
6,weighted,0.5330,0.6068,0.5630,4697
